# Furkan's Notebook: Winning Ticket for Convolution 4

In this notebook, we focus on training a Convolution-4 (Conv-4) network on the CIFAR-10 dataset and investigate the effect of iterative pruning on model performance. The main objective is to examine whether a significantly smaller and sparser subnetwork can still achieve competitive accuracy compared to the original dense model.

Starting from a fully trained Conv-4 network, we iteratively remove a fraction of the weights based on a predefined pruning criterion, such as weight magnitude. After each pruning step, the remaining weights are reinitialized and retrained while keeping the pruning mask fixed. This process allows us to evaluate whether the pruned subnetwork preserves its learning capability. A subnetwork that reaches similar performance with far fewer parameters is referred to as a winning ticket.



In [ ]:
import random
import numpy as np
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torch.nn.functional as F

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [ ]:
import json, os

RESULTS_PATH = "/content/drive/MyDrive/conv4_results.json"

def load_results(path=RESULTS_PATH):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {} 

def save_results(results, path=RESULTS_PATH):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(results, f, indent=2)
    os.replace(tmp, path) 

In [3]:
import os, time, random
import numpy as np
import torch

def _to_cpu_state_dict(sd):
    return {k: v.detach().cpu() for k, v in sd.items()}

def _to_cpu_masks(masks):
    return {k: v.detach().cpu() for k, v in masks.items()}

def _rng_state():
    st = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.random.get_rng_state(),
    }
    if torch.cuda.is_available():
        st["torch_cuda"] = torch.cuda.get_rng_state_all()
    return st

def _set_rng_state(st):
    random.setstate(st["python"])
    np.random.set_state(st["numpy"])
    torch.random.set_rng_state(st["torch"])
    if torch.cuda.is_available() and "torch_cuda" in st:
        torch.cuda.set_rng_state_all(st["torch_cuda"])

def save_ckpt(path, payload: dict):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    payload = dict(payload)
    payload["timestamp"] = time.time()
    torch.save(payload, tmp)
    os.replace(tmp, path)

def load_ckpt(path, map_location="cpu"):
    return torch.load(path, map_location=map_location)

## Loading the Dataset and Define the Original model used in the paper : Conv4

In [ ]:
test_dataset = torchvision.datasets.CIFAR10(
    root='./data/',
    train=False,
    download=True,
    transform=torchvision.transforms.ToTensor())


train_dataset = torchvision.datasets.CIFAR10(
    root='./data/',
    train=True,
    download=True,
    transform=torchvision.transforms.ToTensor())

from torch.utils.data import random_split
train_dataset,  valid_dataset = random_split(
    train_dataset,
    lengths=[45000, 5000],
    generator=torch.Generator().manual_seed(42) 
)


100%|██████████| 170M/170M [00:13<00:00, 12.4MB/s]


In [ ]:
import torch
import torch.nn as nn

class Conv4(nn.Module):
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()

        # Conv-4: (64,64,pool) -> (128,128,pool)
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # 32 -> 16

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # 16 -> 8
        )

        # CIFAR10 128 * 8 * 8 = 8192
        self.classifier = nn.Sequential(
            nn.Flatten(1),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


For the Conv-4 architecture, we use 3×3 convolution layers with padding = 1, which preserves spatial resolution within each convolutional block so that downsampling is performed only by the pooling layers. The network follows a standard (64, 64, pool) → (128, 128, pool) structure. For CIFAR-10 inputs of size 32×32, this results in a feature map of size 128×8×8, which is explicitly flattened and passed to a fully connected classifier with layer sizes 256, 256, and 10, matching the intended Conv-4 design.

## Training the model with the parameters used in the Paper

In [ ]:
BATCH_SIZE = 60
TEST_BATCH_SIZE = 512
LEARNING_RATE = 3e-4

In [7]:
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2)

train_dataloader_adam = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2)

valid_dataloader = DataLoader(
    dataset=valid_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=2)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=2)

## Figure 1 : Pruning , Evaluation and Training with Masks

In [ ]:
#Prunable parameters, mask application, and counting remaining weights

import copy
import torch
import torch.nn as nn

def iter_prunable_params(model: nn.Module):
    # prune conv/linear weights only 
    for name, p in model.named_parameters():
        if p.requires_grad and name.endswith(".weight") and p.dim() >= 2:
            yield name, p

@torch.no_grad()
def apply_masks(model, masks):
    for name, p in model.named_parameters():
        if name in masks:
            p.mul_(masks[name])

def masks_keep_ratio(masks):
    kept = sum(int(m.sum().item()) for m in masks.values())
    total = sum(m.numel() for m in masks.values())
    return kept / total


In [9]:
#Random mask (global)
@torch.no_grad()
def make_global_random_masks(model, keep_ratio: float, seed: int):
    params = list(iter_prunable_params(model))
    device = params[0][1].device
    total = sum(p.numel() for _, p in params)
    k = max(1, int(keep_ratio * total))

    g = torch.Generator(device=device).manual_seed(seed)
    idx = torch.randperm(total, generator=g, device=device)[:k]

    masks = {}
    offset = 0
    for name, p in params:
        n = p.numel()
        mask_flat = torch.zeros(n, device=device, dtype=p.dtype)
        local = idx[(idx >= offset) & (idx < offset + n)] - offset
        mask_flat[local] = 1.0
        masks[name] = mask_flat.view_as(p)
        offset += n
    return masks

In [ ]:
# Magnitude pruning 

def is_conv_weight(name, param):
    return param.dim() == 4  # Conv2d weight: (out,in,k,k)

def is_fc_weight(name, param):
    return param.dim() == 2  # Linear weight: (out,in)

@torch.no_grad()
def prune_by_magnitude_inplace(model, masks, conv_prune_frac=0.10, fc_prune_frac=0.20):
    # prune additional fraction among CURRENTLY-KEPT weights
    for name, p in iter_prunable_params(model):
        w = p.detach()
        m = masks[name].detach()

        # consider only currently kept weights
        kept_vals = w.abs()[m.bool()]
        if kept_vals.numel() == 0:
            continue

        prune_frac = conv_prune_frac if is_conv_weight(name, p) else (fc_prune_frac if is_fc_weight(name, p) else 0.0)
        if prune_frac <= 0:
            continue

        k_prune = int(prune_frac * kept_vals.numel())
        if k_prune < 1:
            continue

        # threshold: prune the smallest magnitudes among kept weights
        thresh = torch.kthvalue(kept_vals, k_prune).values
        to_prune = (w.abs() <= thresh) & (m > 0)

        new_m = m.clone()
        new_m[to_prune] = 0.0
        masks[name] = new_m.to(p.dtype)

    # enforce immediately
    apply_masks(model, masks)
    return masks

In [ ]:
# Training that returns “early-stop iteration” = argmin val loss

@torch.no_grad()
def evaluate_model(model, loader, device, criterion):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        bs = x.size(0)
        total_loss += loss.item() * bs
        total_correct += (logits.argmax(1) == y).sum().item()
        total += bs
    return total_loss / total, total_correct / total

def train_masked_and_get_best_iter(model, train_loader, val_loader, test_loader, device,
                                  optimizer, max_steps, masks=None):
    criterion = nn.CrossEntropyLoss()
    model.to(device)
    if masks is not None:
        apply_masks(model, masks)

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    best_iter = 0

    step = 0
    model.train()
    train_iter = iter(train_loader)

    while step < max_steps:
      if(step % 1000 == 0) :
       print(f"  step {step}/{max_steps} ...")
      try:
            x, y = next(train_iter)
      except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)

      x, y = x.to(device), y.to(device)
      optimizer.zero_grad(set_to_none=True)
      logits = model(x)
      loss = criterion(logits, y)
      loss.backward()
      optimizer.step()
      if masks is not None:
          apply_masks(model, masks)

      step += 1

      if step % 100 == 0 or step == max_steps:
          vloss, _ = evaluate_model(model, val_loader, device, criterion)
          if vloss < best_val_loss:
                best_val_loss = vloss
                best_iter = step
                best_state = copy.deepcopy(model.state_dict())

    # test accuracy at the best iteration
    model.load_state_dict(best_state)
    _, test_acc = evaluate_model(model, test_loader, device, nn.CrossEntropyLoss())
    return best_iter, test_acc

### produce the two curves (random vs winning ticket)

In [ ]:
def run_random_curve_resume(
    model_ctor, keep_ratios, trials,
    train_loader, val_loader, test_loader,
    device, max_steps, lr,
    results_path=RESULTS_PATH,
    resume: bool = True,       
    overwrite: bool = False,      
    seed_base: int = 1000
):
    if resume:
        results = load_results(results_path)
    else:
        results = {}  

    results.setdefault("random", {})

    if overwrite:
        results["random"] = {} 


    for kr in keep_ratios:

      if kr > 0.4 :
        max_steps = max_steps
      elif kr < 0.4 and kr > 0.15 :
         max_steps = 15000
         trials = 6
      elif kr < 0.15 :
         max_steps = 25000
         trials = 6

      for t in range(trials):
            key = f"kr={kr:.6f}_trial={t}"

            if resume and key in results["random"]:
                print(f"[SKIP] random {key} (déjà calculé)")
                continue

            print(f"[RUN] random keep_ratio={kr:.3f} trial {t+1}/{trials}")
            model = model_ctor().to(device)
            masks = make_global_random_masks(model, keep_ratio=kr, seed=seed_base + t)
            opt = torch.optim.Adam(model.parameters(), lr=lr)

            best_iter, test_acc = train_masked_and_get_best_iter(
                model, train_loader, val_loader, test_loader, device,
                optimizer=opt, max_steps=max_steps, masks=masks
            )

            results["random"][key] = {
                "keep_ratio": float(kr),
                "trial": int(t),
                "best_iter": int(best_iter),
                "test_acc": float(test_acc),
                "max_steps": int(max_steps),
                "lr": float(lr),
                "seed": int(seed_base + t),
            }
            save_results(results, results_path)

    return results

In [ ]:
def run_ticket_curve_resume(
    model_ctor, keep_ratios, trials,
    train_loader, val_loader, test_loader,
    device, max_steps, lr, conv_prune=0.15, fc_prune=0.20,
    results_path=RESULTS_PATH,
    resume: bool = True,        
    overwrite: bool = False      
):
    if resume:
        results = load_results(results_path)
    else:
        results = {}  # ignore previous file entirely

    results.setdefault("ticket", {})

    if overwrite:
        results["ticket"] = {}   

    for kr_target in keep_ratios:

       if kr_target > 0.4 :
        max_steps = max_steps
       elif kr_target < 0.4 and kr_target > 0.15 :
         max_steps = 15000
       elif kr_target < 0.15 :
        max_steps = 25000


       for t in range(trials):
            key = f"kr={kr_target:.6f}_trial={t}"

            if resume and key in results["ticket"]:
                print(f"[SKIP] ticket {key} (déjà calculé)")
                continue

            print(f"[RUN] ticket keep_ratio={kr_target:.3f} trial {t+1}/{trials}")

            model0 = model_ctor().to(device)
            init_state = {k: v.detach().clone() for k, v in model0.state_dict().items()}
            masks = {name: torch.ones_like(p) for name, p in iter_prunable_params(model0)}

            current_keep = masks_keep_ratio(masks)
            model = model_ctor().to(device)

            rounds = 0
            while current_keep > kr_target:
                rounds += 1
                print(f"round number {rounds} ")
                model.load_state_dict(init_state)
                apply_masks(model, masks)

                opt = torch.optim.Adam(model.parameters(), lr=lr)
                _best_iter, _test_acc = train_masked_and_get_best_iter(
                    model, train_loader, val_loader, test_loader, device,
                    optimizer=opt, max_steps=max_steps, masks=masks
                )

                masks = prune_by_magnitude_inplace(model, masks, conv_prune, fc_prune)
                current_keep = masks_keep_ratio(masks)

                if current_keep <= kr_target:
                    break

            model.load_state_dict(init_state)
            apply_masks(model, masks)
            opt = torch.optim.Adam(model.parameters(), lr=lr)
            best_iter, test_acc = train_masked_and_get_best_iter(
                model, train_loader, val_loader, test_loader, device,
                optimizer=opt, max_steps=max_steps, masks=masks
            )

            results["ticket"][key] = {
                "keep_ratio": float(kr_target),
                "trial": int(t),
                "rounds": int(rounds),
                "best_iter": int(best_iter),
                "test_acc": float(test_acc),
                "max_steps": int(max_steps),
                "lr": float(lr),
                "conv_prune": float(conv_prune),
                "fc_prune": float(fc_prune),
            }

            save_results(results, results_path)

    return results

In [14]:
# Plotting

import matplotlib.pyplot as plt
import numpy as np

def plot_figure(random_res, ticket_res):
    # This function plots the random and winning ticket curves.
    # random_res/ticket_res are lists of (kr, iters_tensor, accs_tensor)
    x = np.array([kr*100 for kr,_,_ in random_res])

    # left: best_iter (K)
    plt.figure()
    for label, res, ls in [("random", random_res, "--"), ("Conv-4 ticket", ticket_res, "-")]:
        mean_iters = np.array([r[1].mean().item() for r in res]) / 1000.0
        std_iters  = np.array([r[1].std(unbiased=False).item() for r in res]) / 1000.0
        plt.errorbar(x, mean_iters, yerr=std_iters, linestyle=ls, marker="o", capsize=3, label=label)
    plt.xlabel("Percent of Weights Remaining")
    plt.ylabel("Iteration of Minimum Validation Loss (K)")
    plt.gca().invert_xaxis()
    plt.legend()
    plt.show()

    # right: test accuracy at that iteration
    plt.figure()
    for label, res, ls in [("random", random_res, "--"), ("Conv-4 ticket", ticket_res, "-")]:
        mean_acc = np.array([r[2].mean().item() for r in res])
        std_acc  = np.array([r[2].std(unbiased=False).item() for r in res])
        plt.errorbar(x, mean_acc, yerr=std_acc, linestyle=ls, marker="o", capsize=3, label=label)
    plt.xlabel("Percent of Weights Remaining")
    plt.ylabel("Test Accuracy at Best-Validation Iteration")
    plt.gca().invert_xaxis()
    plt.legend()
    plt.show()

In [15]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [16]:
RESULTS_PATH = "/content/conv4_results_win.json"


In [ ]:
import os

RESULTS_PATH = "/content/conv4_results.json"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

model_ctor = lambda: Conv4(in_channels=3, num_classes=10)

keep_ratios = [1.0, 0.412, 0.170, 0.071, 0.030, 0.013]
max_steps = 8000

random_trials = 5
ticket_trials = 2

conv_prune_frac = 0.10
fc_prune_frac   = 0.20

set_seed(123)
random_res = run_random_curve_resume(
    model_ctor=model_ctor,
    keep_ratios=keep_ratios,
    trials=random_trials,
    train_loader=train_dataloader,
    val_loader=valid_dataloader,
    test_loader=test_dataloader,
    device=device,
    max_steps=max_steps,
    lr=LEARNING_RATE,
    results_path=RESULTS_PATH, 
    #overwrite=True,          
)


[RUN] random keep_ratio=1.000 trial 1/5
  step 0/8000 ...
  step 1000/8000 ...
  step 2000/8000 ...
  step 3000/8000 ...
  step 4000/8000 ...
  step 5000/8000 ...
  step 6000/8000 ...
  step 7000/8000 ...
[RUN] random keep_ratio=1.000 trial 2/5
  step 0/8000 ...
  step 1000/8000 ...
  step 2000/8000 ...
  step 3000/8000 ...
  step 4000/8000 ...
  step 5000/8000 ...
  step 6000/8000 ...
  step 7000/8000 ...
[RUN] random keep_ratio=1.000 trial 3/5
  step 0/8000 ...
  step 1000/8000 ...
  step 2000/8000 ...
  step 3000/8000 ...
  step 4000/8000 ...
  step 5000/8000 ...
  step 6000/8000 ...
  step 7000/8000 ...
[RUN] random keep_ratio=1.000 trial 4/5
  step 0/8000 ...
  step 1000/8000 ...
  step 2000/8000 ...
  step 3000/8000 ...
  step 4000/8000 ...
  step 5000/8000 ...
  step 6000/8000 ...
  step 7000/8000 ...
[RUN] random keep_ratio=1.000 trial 5/5
  step 0/8000 ...
  step 1000/8000 ...
  step 2000/8000 ...
  step 3000/8000 ...
  step 4000/8000 ...
  step 5000/8000 ...
  step 6000/8000 

In [ ]:
set_seed(456)
ticket_res = run_ticket_curve_resume(
    model_ctor=model_ctor,
    keep_ratios=keep_ratios,
    trials=ticket_trials,
    train_loader=train_dataloader,
    val_loader=valid_dataloader,
    test_loader=test_dataloader,
    device=device,
    max_steps=max_steps,
    lr=LEARNING_RATE,
    conv_prune=conv_prune_frac,
    fc_prune=fc_prune_frac,
    results_path=RESULTS_PATH,
)

[RUN] ticket keep_ratio=1.000 trial 1/2
  step 0/8000 ...
  step 1000/8000 ...
  step 2000/8000 ...
  step 3000/8000 ...
  step 4000/8000 ...
  step 5000/8000 ...
  step 6000/8000 ...
  step 7000/8000 ...
[RUN] ticket keep_ratio=1.000 trial 2/2
  step 0/8000 ...
  step 1000/8000 ...
  step 2000/8000 ...
  step 3000/8000 ...
  step 4000/8000 ...
  step 5000/8000 ...
  step 6000/8000 ...
  step 7000/8000 ...
[RUN] ticket keep_ratio=0.412 trial 1/2
round number 1 
  step 0/8000 ...
  step 1000/8000 ...
  step 2000/8000 ...
  step 3000/8000 ...
  step 4000/8000 ...
  step 5000/8000 ...
  step 6000/8000 ...
  step 7000/8000 ...
round number 2 
  step 0/8000 ...
  step 1000/8000 ...
  step 2000/8000 ...
  step 3000/8000 ...
  step 4000/8000 ...
  step 5000/8000 ...
  step 6000/8000 ...
  step 7000/8000 ...
round number 3 
  step 0/8000 ...
  step 1000/8000 ...
  step 2000/8000 ...
  step 3000/8000 ...
  step 4000/8000 ...
  step 5000/8000 ...
  step 6000/8000 ...
  step 7000/8000 ...
round n

In [ ]:
model_ctor = lambda: Conv4(in_channels=3, num_classes=10)

keep_ratios = [0.071, 0.030, 0.013]
max_steps = 8000

random_trials = 5
ticket_trials = 2

conv_prune_frac = 0.10
fc_prune_frac   = 0.20

In [ ]:
set_seed(456)
ticket_res = run_ticket_curve_resume(
    model_ctor=model_ctor,
    keep_ratios=keep_ratios,
    trials=ticket_trials,
    train_loader=train_dataloader,
    val_loader=valid_dataloader,
    test_loader=test_dataloader,
    device=device,
    max_steps=max_steps,
    lr=LEARNING_RATE,
    conv_prune=conv_prune_frac,
    fc_prune=fc_prune_frac,
    results_path=RESULTS_PATH,
)

[RUN] ticket keep_ratio=0.071 trial 1/2
round number 1 
  step 0/25000 ...
  step 1000/25000 ...
  step 2000/25000 ...
  step 3000/25000 ...
  step 4000/25000 ...
  step 5000/25000 ...
  step 6000/25000 ...
  step 7000/25000 ...
  step 8000/25000 ...
  step 9000/25000 ...
  step 10000/25000 ...
  step 11000/25000 ...
  step 12000/25000 ...
  step 13000/25000 ...
  step 14000/25000 ...
  step 15000/25000 ...
  step 16000/25000 ...
  step 17000/25000 ...
  step 18000/25000 ...
  step 19000/25000 ...
  step 20000/25000 ...
  step 21000/25000 ...
  step 22000/25000 ...
  step 23000/25000 ...
  step 24000/25000 ...
round number 2 
  step 0/25000 ...
  step 1000/25000 ...
  step 2000/25000 ...
  step 3000/25000 ...
  step 4000/25000 ...
  step 5000/25000 ...
  step 6000/25000 ...
  step 7000/25000 ...
  step 8000/25000 ...
  step 9000/25000 ...
  step 10000/25000 ...
  step 11000/25000 ...
  step 12000/25000 ...
  step 13000/25000 ...
  step 14000/25000 ...
  step 15000/25000 ...
  step 1600

KeyboardInterrupt: 

In [ ]:
def process_results_for_plot(raw_results_dict):
    processed = []
    grouped_by_kr = {}
    for key, val in raw_results_dict.items():
        kr = val["keep_ratio"]
        if kr not in grouped_by_kr:
            grouped_by_kr[kr] = {"best_iters": [], "test_accs": []}
        grouped_by_kr[kr]["best_iters"].append(val["best_iter"])
        grouped_by_kr[kr]["test_accs"].append(val["test_acc"])

    for kr in sorted(grouped_by_kr.keys(), reverse=True):
        iters = torch.tensor(grouped_by_kr[kr]["best_iters"], dtype=torch.float32)
        accs = torch.tensor(grouped_by_kr[kr]["test_accs"], dtype=torch.float32)
        processed.append((kr, iters, accs))
    return processed

processed_random_res = process_results_for_plot(random_res["random"])
processed_ticket_res = process_results_for_plot(ticket_res["ticket"])

plot_figure(processed_random_res, processed_ticket_res)

In [ ]:
print("RANDOM:")
for kr, iters, accs in processed_random_res:
    print(f"keep={kr:.3f} | best_iter mean={iters.mean():.1f} std={iters.std(unbiased=False):.1f} | "
          f"test_acc mean={accs.mean():.4f} std={accs.std(unbiased=False):.4f}")

print("\nTICKET:")
for kr, iters, accs in processed_ticket_res:
    print(f"keep={kr:.3f} | best_iter mean={iters.mean():.1f} std={iters.std(unbiased=False):.1f} | "
          f"test_acc mean={accs.mean():.4f} std={accs.std(unbiased=False):.4f}")

Due to multiple executions of this notebook during development and debugging, some runs were interrupted early, which may result in occasional stop or runtime errors in certain cells. These interruptions do not affect the validity of the reported results. To ensure consistency and clarity, all final experimental outcomes were collected, verified, and consolidated in a separate results file, where the complete and stable measurements are presented.